In [1]:
import rasterio
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, FixedLocator, FixedFormatter
import numpy as np
import xarray as xr
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling
import os
import glob
from matplotlib.colors import ListedColormap
from datetime import datetime
import pandas as pd
from netCDF4 import Dataset
from scipy.interpolate import interp1d

In [2]:
def convert_tif_netcdf_lai(filepath):
    geotiff_files = sorted(glob.glob(filepath + '*'))
    if len(geotiff_files) == 0:
        raise FileNotFoundError("No GeoTIFF files found")

    data_list = []
    time_list = []

    # --- Reference grid from first file ---
    with rasterio.open(geotiff_files[0]) as src0:
        height, width = src0.height, src0.width
        transform = src0.transform
        crs = src0.crs
        nodata = src0.nodata

        # Pixel-center lon/lat (ASSUMES EPSG:4326)
        cols = np.arange(width)
        rows = np.arange(height)
        # print(cols, rows)

        lon = transform.c + (cols + 0.5) * transform.a
        lat = transform.f + (rows + 0.5) * transform.e  # usually decreasing

    for geotiff_file in geotiff_files:
        with rasterio.open(geotiff_file) as src:
            # Safety checks
            if (src.height, src.width) != (height, width):
                raise ValueError(f"Shape mismatch in {geotiff_file}")
            if src.transform != transform:
                raise ValueError(f"Transform mismatch in {geotiff_file}")
            if src.crs != crs:
                raise ValueError(f"CRS mismatch in {geotiff_file}")

            arr = src.read(1).astype("float32")

            if nodata is not None:
                arr[arr == nodata] = np.nan

            
            time_str = os.path.basename(geotiff_file).split('_')[1].split('.')[0]
            time_list.append(pd.to_datetime(time_str))

            data_list.append(arr)

    data = np.stack(data_list, axis=0)

    da = xr.DataArray(
        data,
        dims=["XTIME", "lat", "lon"],
        coords={
            "XTIME": ("XTIME", np.array(time_list, dtype="datetime64[ns]")),
            "lat": ("lat", lat),
            "lon": ("lon", lon),
        },
        name="LAI",
    )

    ds = xr.Dataset({"LAI": da})

    # Metadata
    ds["LAI"].attrs.update({
        "long_name": "Leaf Area Index",
        "units": "m^2/m^2",
    })
    ds.attrs.update({
        "crs": str(crs),
        "note": "Native MODIS LAI grid; no reprojection or resampling applied",
    })

    return ds


In [3]:
def interpolate_netcdf_daily(nc_data, output_file, time_var_name, data_var_name, time_format=None):
    """
    Interpolates NetCDF data to a daily time resolution and saves to a new NetCDF file.

    Parameters:
    - input_file (str): Path to the input NetCDF file.
    - output_file (str): Path to the output NetCDF file.
    - time_var_name (str): Name of the time variable in the NetCDF file.
    - data_var_name (str): Name of the data variable to interpolate.
    - time_format (str): Format string for converting time to pandas datetime. If None, assume the time is already in datetime format.

    Returns:
    - None
    """
    
    
    time = nc_data.variables[time_var_name][:]
    data = nc_data.variables[data_var_name][:]
    
    
    if time_format:
       
        time_dates = pd.to_datetime(time, format=time_format)
    else:
        #
        try:
            time_dates = pd.to_datetime(time)
        except Exception as e:
            raise ValueError(f"Failed to convert time to datetime: {e}")
    
    nc_data.close()
    
    start_date = time_dates.min()
    end_date = time_dates.max()
    full_dates = pd.date_range(start=start_date, end=end_date, freq='D')

    # Convert original dates to numeric values for interpolation
    time_numeric = pd.to_numeric(time_dates)

   
    interp_func = interp1d(time_numeric, data, kind='linear', axis=0, fill_value='extrapolate')

    
    full_dates_numeric = pd.to_numeric(full_dates)

    # Interpolate data
    interpolated_data = interp_func(full_dates_numeric)

    
    data_array = xr.DataArray(interpolated_data, dims=['XTIME', 'south_north', 'west_east'], coords={'XTIME': full_dates})
    dataset = xr.Dataset({'LAI': data_array})

    return dataset


In [4]:
# input_file = convert_tif_netcdf_lai('/bsuscratch/stanleyakor/swe_emulator/modis/lai/')
# output_file = '../data/lai.nc'
# time_var_name = 'XTIME'
# data_var_name = 'LAI'




In [5]:
# input_file.to_netcdf('../data/lai_unregrided.nc')

In [6]:
# xr.open_dataset('../data/lai_unregrided.nc')['LAI'].isel(XTIME = 570).plot()

# -> before interpolating to gap fill 7 days, regrid to match wrf dims, new file = lai_regrided.nc using regrid_3.ipynb

In [7]:
# input_file = xr.open_dataset('../data/lai_regrided.nc')['LAI']

In [8]:
# data_Var = interpolate_netcdf_daily(input_file, output_file, time_var_name,data_var_name)

In [9]:
# data_Var.to_netcdf(output_file)

In [10]:
# ds = xr.open_dataset('../data/lai.nc')

In [11]:
# ds.isel(XTIME = 285)['LAI'].plot()